Mahan Madani - 830504035

## Table of Contents

- [Import Libraries and Configs](#import-libraries-and-configs)
- [Datasets and Dataloader](#datasets-and-dataloaders)
- [Find and Index CLIP Embeddings Using FAISS](#find-and-index-clip-embeddings-using-faiss)
- [Create Metadata Files to Store Image Captions](#create-metadata-files-to-store-image-captions)
- [Similar Image Retrieval Sample](#similar-image-retrieval-sample)
- [Load FAISS index and JSON Metadata (for future runs)](#load-faiss-index-and-json-metadata-for-future-runs)

## Import libraries and Configs

In [15]:
import os
import json
import faiss
import torch
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from torch.utils.data import DataLoader
from transformers import CLIPTokenizerFast, CLIPImageProcessorFast

In [16]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TRAIN_CAPTIONS_PATH,
                            TEST_IMAGE_DIR,
                            TEST_CAPTIONS_PATH,
                            FAISS_CAPTION_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CAPTION_METADATA_PATH,
                            CLIP_MODEL_NAME)

from Modules.retrieval_module import Retriever
from Modules.datasets import CLIPCaptionDataset, CLIPCaptionDataCollator, CLIPImageDataset, CLIPImageDataCollator
from Modules.embedding_module import Embedder

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## Datasets and Dataloaders

Place the flickr30k dataset in the dataset folder and run the data_preprocess notebook to split it into train-test subsets.

In [ ]:
train_dataset = CLIPCaptionDataset(TRAIN_CAPTIONS_PATH)
test_dataset  = CLIPCaptionDataset(TEST_CAPTIONS_PATH)

tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
collator = CLIPCaptionDataCollator(tokenizer, DEVICE)

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collator)

## Find and Index CLIP Embeddings Using FAISS

In [7]:
embedder = Embedder(model_name=CLIP_MODEL_NAME, use_local_files=True, device=DEVICE)
faiss_caption_index = faiss.IndexFlatIP(embedder.embedding_dim)  # use cosine similarity

In [11]:
caption_metadata = {}
caption_index = 0

In [ ]:
loader = tqdm(train_loader, desc="Extracting CLIP embeddings")

for batch in loader:
    input_ids = batch.get('input_ids')
    attention_mask = batch.get('attention_mask')
    captions = batch.get('captions')
    image_names = batch.get('image_names')
    
    embeddings = embedder.get_text_embedding(input_ids, attention_mask)
    faiss_caption_index.add(embeddings)
    
    for i in range(len(captions)):
        caption_metadata[caption_index] = {
            "image_name": image_names[i],
            "caption": captions[i],
        }
        caption_index += 1

Extracting CLIP embeddings: 100%|██████████| 4810/4810 [00:20<00:00, 235.54it/s]


In [12]:
faiss.write_index(faiss_caption_index, FAISS_CAPTION_PATH)
print(f"Indexed {faiss_caption_index.ntotal} captions")

Indexed 153915 captions


In [13]:
with open(CAPTION_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(caption_metadata, f, indent=2, ensure_ascii=False)

## Update Train and Test Metadata Files

In [34]:
retriever_caption = Retriever(metadata_path=CAPTION_METADATA_PATH, faiss_path=FAISS_CAPTION_PATH)

In [35]:
train_dataset = CLIPImageDataset(TRAIN_IMAGE_DIR, TRAIN_CAPTIONS_PATH)
test_dataset  = CLIPImageDataset(TEST_IMAGE_DIR, TEST_CAPTIONS_PATH)

processor = CLIPImageProcessorFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
collator = CLIPImageDataCollator(processor, DEVICE)

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collator)

In [36]:
with open(TRAIN_METADATA_PATH, "r", encoding="utf-8") as f:
    train_metadata = json.load(f)
    
with open(TEST_METADATA_PATH, "r", encoding="utf-8") as f:
    test_metadata = json.load(f)
    
train_index = 0
test_index = 0

In [37]:
loader = tqdm(train_loader, desc="Finding similar captions")

for batch in loader:
    images = batch['pixel_values'].to(DEVICE) # shape [B, 3, W, H]
    image_names = batch['image_names']
    
    embeddings = embedder.get_image_embedding(images)
    similar_captions_data = retriever_caption.retrieve_similar_from_metadata(embeddings, k=10)

    for i in range(len(images)):
        similar_captions = []
        for caption_data in similar_captions_data[i]:
            if caption_data['image_name'] != image_names[i]:
                similar_captions.append(caption_data['caption'])
                 
                if len(similar_captions) >= 5:
                    break
                
        train_metadata[str(train_index)].update({
            "similar_captions": similar_captions
        })
        train_index += 1

Finding similar captions: 100%|██████████| 962/962 [04:26<00:00,  3.62it/s]


In [31]:
loader = tqdm(test_loader, desc="Finding similar captions")

for batch in loader:
    images = batch['pixel_values'].to(DEVICE) # shape [B, 3, W, H]
    image_names = batch['image_names']
    
    embeddings = embedder.get_image_embedding(images)
    similar_captions_data = retriever_caption.retrieve_similar_from_metadata(embeddings, k=5)

    for i in range(len(images)):
        similar_captions = []
        for caption_data in similar_captions_data[i]:
            similar_captions.append(caption_data['caption'])

        test_metadata[str(test_index)].update({
            "similar_captions": similar_captions
        })
        test_index += 1

Finding similar captions: 100%|██████████| 32/32 [00:08<00:00,  3.83it/s]


Save JSON Metadata

In [ ]:
with open(TRAIN_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(train_metadata, f, indent=2, ensure_ascii=False)
    
with open(TEST_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(test_metadata, f, indent=2, ensure_ascii=False)

## Similar Image Retrieval Sample

In [ ]:
# Function to display query image and the k most similar images
def display_images(query_image, retrieved_images):    
    n = len(retrieved_images) + 1
    plt.figure(figsize=(15, 4))

    plt.subplot(1, n, 1)
    plt.imshow(query_image)
    plt.axis('off') 
    plt.title("Test Image")

    for i in range(n-1):
        plt.subplot(1, n, i+2)
        image = retrieved_images[i]
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Retrieved Image {i+1}")

    plt.tight_layout()
    plt.show()

In [ ]:
retriever = Retriever(FAIS_IMAGE_PATH, TRAIN_METADATA_PATH)

In [ ]:
sample_idx = [1, 112, 501, 777, 910]
k = 4

pixel_values = collator([test_dataset[idx] for idx in sample_idx])['pixel_values']
embeddings = embedder.get_image_embedding(pixel_values)
similar_images_idx = retriever.retrieve_similar_indices(embeddings, k=k)

for i in range(len(sample_idx)):
    query_image = test_dataset[sample_idx[i]]['image']
    similar_images = []
    for idx in similar_images_idx[i]:
        similar_images.append(train_dataset[idx]['image'])

    display_images(query_image, similar_images)

Sample Captions

In [ ]:
query_image_metadata = test_metadata[str(sample_idx[-1])]
print("Query image captions:")
print(*query_image_metadata['captions'], sep='\n')

print("\nRetrieved Image Captions:")
for idx in query_image_metadata['similar_images'][:k]:
    similar_image_metadata = train_metadata[str(idx)]
    print(similar_image_metadata['image_name'])
    print(*similar_image_metadata['captions'], sep='\n', end='\n\n')

## Load FAISS index and JSON Metadata (for future runs)

In [ ]:
# faiss_index = faiss.read_index(FAISS_PATH)
faiss_index = faiss.read_index(FAISS_CAPTION_PATH)
print(f"Number of vectors in index: {faiss_index.ntotal}")

In [ ]:
with open(TRAIN_METADATA_PATH, "r", encoding="utf-8") as f:
    train_metadata = json.load(f)
    
with open(TEST_METADATA_PATH, "r", encoding="utf-8") as f:
    test_metadata = json.load(f)

print(train_metadata["0"])  # keys: 'title', 'captions', etc.